<a href="https://colab.research.google.com/github/Pramila003/Vectorless_Rag/blob/main/vectorless_rag_using_pageindex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U pageindex openai python-dotenv

  Using cached openai-3.31.0-py3-none-any.whl.metadata (44 kB)


In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = ""
OPENAI_API_KEY    = ""

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("OpenAI key loaded:   ", "✅" if OPENAI_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
OpenAI key loaded:    ✅


In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = ""
GROQ_API_KEY      = ""

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:     ", "✅" if GROQ_API_KEY      else "❌ Missing!")


from pageindex import PageIndexClient
from openai import OpenAI

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

# 👈 2. Add base_url so OpenAI's SDK redirects calls to Groq
openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY
)

PageIndex key loaded: ✅
Groq key loaded:      ✅


In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ PageIndex client ready")
print("✅ OpenAI client ready")

✅ PageIndex client ready
✅ OpenAI client ready


In [ ]:
PDF_PATH = r"/content/ITSM Operations Runbook.pdf"

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: /content/ITSM Operations Runbook.pdf
✅ Uploaded!
📋 Document ID: pi-cmv9balms00420cnwjfz57m6f
   (Save this ID — you'll use it throughout the notebook)


/tmp/ipykernel_589/2279511594.py:4: UserWarning: Document "ITSM Operations Runbook.pdf" was stored as "ITSM Operations Runbook_1.pdf".
  result = pi_client.submit_document(PDF_PATH)


In [ ]:
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")

    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break

    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


In [ ]:
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "ITSM Operations Runbook",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 5,
  "summary": "This ITSM Operations Runbook (SOP-ITSM-042) establishes global production standards for the IT service lifecycle, covering document scope, governance, and six core disciplines:\n- **Purpose & Governance:** Outlines protocols for SLA compliance, MTTR reduction, and auditability.\n- **Record-Type Decision Framework:** Distinguishes between unplanned Incidents and planned Service Requests.\n- **Standard Title Syntax:** Mandates strict naming conventions for incident triage and lists prohibited vague titles.\n- **Incident Data Fields:** Specifies mandatory, recommended, conditional, and system-generated fields for record validation.\n- **Incident Triage & Management:** Details step-by-step intake, core diagnostics, deduplication, priority calculation via impact/urgency matrices, JSON schemas, lifecycle states, assignments, SLAs, 

In [ ]:
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] ITSM Operations Runbook  (p.?)
  └─ [0001] ITSM Operations Runbook (intro)  (p.?)
  └─ [0002] 1. Purpose & Core Governance  (p.?)
  └─ [0003] 2. Record-Type Decision Framework  (p.?)
  └─ [0004] 3. Standard Title Format & Syntax Policy  (p.?)
  └─ [0005] 4. Required Incident Data Fields Specification  (p.?)
  └─ [0006] 5. How to Raise an Incident (Step-by-Step Triage)  (p.?)
    └─ [0007] 5. How to Raise an Incident (Step-by-Step Triage) (intro)  (p.?)
    └─ [0008] Step 1: Intake & Scope Verification  (p.?)
    └─ [0009] Step 2: Active Deduplication Protocol  (p.?)
    └─ [0010] Step 3: Determine Impact, Urgency & Priority Calculation  (p.?)
      └─ [0011] Step 3: Determine Impact, Urgency & Priority Calculation (intro)  (p.?)
      └─ [0012] Mandatory Pending Reason Values:  (p.?)
      └─ [0013] Work Note & Trigger Policy:  (p.?)
  └─ [0014] 7. How to Raise and Process a Service Request  (p.?)
  └─ [0015] 8. Escalation Procedure & Operations Proto

In [ ]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 18
   Each node = one retrievable section of the document


In [ ]:
def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.

    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """

    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
query = "How to Raise an Incident (Step-by-Step Triage)"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: How to Raise an Incident (Step-by-Step Triage)

🧠 LLM Reasoning:
The query exactly matches the title of node 0006, which is the container for the step-by-step incident‑raising procedure. The detailed steps are in its children: intro (0007), Step 1 (0008), Step 2 (0009), and Step 3 (0010) with its own intro sub‑node (0011). These nodes together contain the complete answer, so they are returned.

🎯 Selected Node IDs: ['0006', '0007', '0008', '0009', '0010', '0011']


In [ ]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [ ]:
def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."

    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:

    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")

    return answer

In [ ]:
answer = vectorless_rag(
    query=" Required Service Request Fields Specification",
    tree=pageindex_tree
)

🔍 Query:  Required Service Request Fields Specification

🧠 Reasoning: The query asks for the specification of required fields for a Service Request. In the document tree, the section titled '7. How to Raise and Process a Service Request' (node_id 0014) directly addresse...
🎯 Retrieved node IDs: ['0014']
📄 Sections found: ['7. How to Raise and Process a Service Request']

📝 Answer:
**Required Service Request Fields (as defined in the Service Request payload):**

| Field | Description | Required? |
|-------|-------------|-----------|
| `request_number` | Unique identifier for the request (e.g., “REQ1004”). | Yes |
| `requester_id` | ID of the user who raises the request. | Yes |
| `title` | Short description of the request purpose. | Yes |
| `category` | Business area of the request (e.g., “Access Management”). | Yes |
| `status` | Current lifecycle state (e.g., “Pending Approval”). | Yes |
| `approval_required` | Boolean flag indicating if approval is needed. | Yes |
| `approver` (objec